# 中老年人群高血脂症风险预警及干预优化

本 Notebook 从原始附件直接读取数据，依次完成问题一、问题二和问题三。所有患者参数、阈值、约束检查和论文引用数值均可追溯到 `code/outputs/`。


In [1]:
from code.solution import (
    PARAMS, OUTPUT_DIR, build_run_summary, configure_plotting, load_and_preprocess,
    run_problem1, run_problem2, run_problem3, write_json, write_results_report
)
import time
start_time = time.perf_counter()
font_name = configure_plotting()
print(f"绘图字体：{font_name}")


绘图字体：Microsoft YaHei


In [2]:
df, data_quality = load_and_preprocess()
print(data_quality)


数据读取完成：1000 行，痰湿质患者 278 人
诊断标签与四项血脂异常规则一致率：100.0%
{'rows': 1000, 'columns': 45, 'duplicate_sample_ids': 0, 'missing_values': 0, 'label_prevalence': 0.793, 'diagnostic_rule_agreement': 1.0, 'phlegm_constitution_count': 278, 'activity_identity_max_error': 0.0}


In [3]:
res1 = run_problem1(df, PARAMS)
print(res1["endpoint_performance"].round(4).to_string(index=False))
print(res1["screen"].round(4).to_string(index=False))
print(res1["constitution"].round(4).to_string(index=False))
print(res1["constitution_sensitivity"].round(4).to_string(index=False))


问题1：痰湿终点入选 []；非诊断高血脂关联入选 ['血尿酸']；共同指标 []
九体质标签整体LR检验 P=0.562，未调整卡方检验 P=0.7
     终点    指标      均值    标准差  折数
  痰湿严重度  折外R² -0.0564 0.0276  15
高血脂关联筛查 折外AUC  0.8480 0.0269  15
     终点         指标名称  是否诊断泄漏 折外模型指标  折外模型性能均值  折外模型性能标准差  折外置换重要性均值  折外正重要性频率   关联统计量   原始P值  FDR_q值  单指标AUC  是否入选                         未入选原因
  痰湿严重度      总胆固醇_TC   False     R²   -0.0564     0.0276    -0.0041    0.4000 -0.0352 0.2659  0.7612     NaN False 折外R²不高于0；折外重要性不稳定；相关强度或FDR未达标
  痰湿严重度      甘油三酯_TG   False     R²   -0.0564     0.0276    -0.0064    0.1333 -0.0158 0.6186  0.7612     NaN False 折外R²不高于0；折外重要性不稳定；相关强度或FDR未达标
  痰湿严重度 低密度脂蛋白_LDL_C   False     R²   -0.0564     0.0276    -0.0046    0.2000  0.0099 0.7551  0.7612     NaN False 折外R²不高于0；折外重要性不稳定；相关强度或FDR未达标
  痰湿严重度 高密度脂蛋白_HDL_C   False     R²   -0.0564     0.0276    -0.0065    0.2000  0.0139 0.6607  0.7612     NaN False 折外R²不高于0；折外重要性不稳定；相关强度或FDR未达标
  痰湿严重度           血糖   False     R²   -0.0564     0.0276    -0.0022    0.5333 -0.0205 0.5181  0.7612 

In [4]:
res2 = run_problem2(df, PARAMS)
print(res2["candidate_models"].round(4).to_string(index=False))
print(res2["test_tiers"].round(4).to_string(index=False))
print(res2["profile"].round(4).to_string(index=False))
print(res2["combinations"].head(10).round(4).to_string(index=False))


问题2：选用 随机森林，独立测试 AUC=0.821，Brier=0.135
测试集低/中/高管理人数：[21, 247, 32]；当前异常率：[0.0, 0.834, 1.0]
痰湿体质高危核心组合首位：痰湿体质 + 血脂异常 + 痰湿积分≥60
        模型  训练集CV_AUC均值  训练集CV_AUC标准差
      随机森林       0.8653        0.0302
加权Logistic       0.6350        0.0784
  数据集  风险等级编码 风险等级  人数     占比  实际患病率  患病率95%CI下限  患病率95%CI上限
独立测试集       1   低危  21 0.0700  0.000      0.0000      0.1115
独立测试集       2   中危 247 0.8233  0.834      0.7838      0.8764
独立测试集       3   高危  32 0.1067  1.000      0.9251      1.0000
 风险等级编码  痰湿质积分_median  痰湿质积分_mean  活动量表总分_median  活动量表总分_mean  BMI_median  BMI_mean  血糖_median  血糖_mean  血尿酸_median  血尿酸_mean  年龄组_median  年龄组_mean 风险等级                                               特征分层规则
      1          21.0      25.714           52.0       51.095      20.989    21.239      5.002    5.046       255.0   269.429         3.0     3.095   低危                                           未触发高危或中危规则
      2          28.0      29.846           50.0       49.526      22.226    22.140      5.092    5.020  

In [5]:
res3 = run_problem3(df, PARAMS)
print(res3["summary"][res3["summary"]["样本ID"].isin([1,2,3])].round(4).to_string(index=False))
print(res3["plans_123"].round(4).to_string(index=False))
print(res3["matching"].round(4).to_string(index=False))


问题3：已对全部 278 名痰湿质患者求解，约束回代全部通过=True
样本1：64.0→48.5分，成本1014元，最大允许强度1级
样本2：58.0→37.0分，成本1240元，最大允许强度2级
样本3：59.0→33.0分，成本1674元，最大允许强度3级
 样本ID  年龄组  活动量表总分  初始痰湿积分      初始调理等级  最大允许活动强度  推荐首月活动强度  推荐首月每周次数  最终痰湿积分  降低分数   降低比例  最大可降低分数  效果保留率  六个月总成本
    1    2    38.0    64.0   强化调理(≥62)         1         1        10    48.5  15.5 0.2422     17.0 0.9118    1014
    2    1    40.0    58.0   基础调理(≤58)         2         2        10    37.0  21.0 0.3621     23.0 0.9130    1240
    3    1    63.0    59.0 中度调理(59-61)         3         3        10    33.0  26.0 0.4407     28.5 0.9123    1674
 样本ID  月份  月初痰湿积分  调理等级              核心调理方式  活动强度  单次时长_分钟  每周次数  当月调理成本  当月活动成本  当月总成本  当月活动降幅  月末痰湿积分
    1   1    64.0     3 饮食调理+穴位按摩+八段锦+中药代茶饮     1       10    10     130     120    250    0.05    61.0
    1   2    61.0     2       饮食调理+穴位按摩+八段锦     1       10    10      80     120    200    0.05    58.0
    1   3    58.0     1           饮食调理+穴位按摩     1       10    10      30     120    150    0.05    5

In [6]:
elapsed_seconds = time.perf_counter() - start_time
write_results_report(data_quality, res1, res2, res3, elapsed_seconds)
run_summary = build_run_summary(font_name, data_quality, res1, res2, res3, elapsed_seconds)
write_json(OUTPUT_DIR / "run_summary.json", run_summary)
print(f"完整流程通过，耗时 {elapsed_seconds:.1f} 秒。")
print("所有 CSV/JSON 已保存到 code/outputs，PDF 图已保存到 figures。")


完整流程通过，耗时 66.1 秒。
所有 CSV/JSON 已保存到 code/outputs，PDF 图已保存到 figures。
